In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
import scipy.linalg as la
from scipy.linalg import eig
from statsmodels.tsa.vector_ar.var_model import VAR
import statsmodels.api as sm
print(f"modules have been imported.")

In [ ]:
# Step 1: Lets grab the historical time series from Mohaddes first. 
# Question should we grab it by economies first? if yes then lets take the 'Country Data'
print(os.getcwd())
target_filename = "Country Data (1979Q2-2023Q3).xls"
target_file_path = os.getcwd() + "\\" + "GVAR Database (1979Q2-2023Q3)\\" + target_filename
print(f"target_file_path is: {target_file_path}")
# dataframe the country data
countries_df = pd.read_excel(target_file_path, sheet_name = None)
countries_df.keys()


In [ ]:
us_df = countries_df['USA']
# convert the date into datetime format 
# ie: 1979Q2 is 30-Jun-1979.
us_df['date_2'] = pd.PeriodIndex(us_df['date'], freq = 'Q').to_timestamp(how = "end").date

target_cols = [col for col in us_df.columns if 'date' not in col]
target_cols = ['date_2'] + target_cols 
print(target_cols)
us_df = us_df[target_cols]
us_df = us_df.rename(columns={'date_2': 'date'})
us_df

In [ ]:
# 1. This is to test the weak exogeneity for each variable components in the us_df table. 
# 2. We need to do the AIC and BIC test to find out the lags that are most optimal for the VARX


In [ ]:
# to run the VARX model.
# After running the weak exogeneity test, we will then proceed to run the VARX for
# the variables are do not violate the weak exogeneity test. 
endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
display(us_exog_df)

# lets assume that our lags are as follows p_{domestic} = 2, q_{foreign} = 1
# so we are now finding for VAR(p, q) = VAR(2, 1)
# so for our exog data we need to start at iloc[1:] as our t 
# and shift(1).iloc[1:] as our t - 1
# for endog_data we will start at us_df.iloc[2:] as our t 
# and us_df.shift(1).iloc[2:] as our t - 2

p, q = 2, 1

# to define the exog data here based on lags (defined by AIC/BIC/HQIC) and shifts (based on also lags?).
us_exog_contemp_df = us_exog_df.iloc[p:] # align to match the loss of 2 domestic lags.
us_exog_lag1_df = us_exog_df.shift(1).iloc[p:]
display(us_exog_contemp_df)
display(us_exog_lag1_df)
print(f"us_exog_df shift(1) is: {display(us_exog_df.shift(1))}")
print(f"us_exog_df shift(1).iloc[p:] is: {display(us_exog_df.shift(1).iloc[p:])}")

# Concatenate the exog data here with the shifts and lags. 
us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df], axis = 1)
print(f"us_exog_final_df is: {display(us_exog_final_df)}")
col_names = [f"{col}_lag1" for col in us_exog_df.columns]
all_col_exog_names = us_exog_df.columns.tolist() + col_names
print(all_col_exog_names)
# This is to ensure that the naming convention for the lags and the contemporary data are defined correctly.
us_exog_final_df.columns = all_col_exog_names
print(f"us_exog_final_df is: {display(us_exog_final_df)}")

# Add in the constant from the exog data. 
us_exog_final_df = sm.add_constant(us_exog_final_df)
print(f"us_exog_final_df after including constant is: {display(us_exog_final_df)}")

# define us endogeneous data
us_endog_final_df = us_endog_df.iloc[p:]
display(us_endog_final_df)

# Fit the level VARX model via statsmodels 
# we used maxlags = max(p,q) for the domestic variables and passing foreign vectors as exog.
varx_model = VAR(endog = us_endog_final_df, exog = us_exog_final_df)
varx_results = varx_model.fit(maxlags = p)

print(f"--- US VARX({p},{q}) Model Estimated Successfully ---")
print(f"Endogenous Matrix Dimensions (Variables): {us_endog_final_df.shape[1]}")
print(f"Exogenous Matrix Dimensions (Regressors): {us_exog_final_df.shape[1]}")
print(f"\nCoefficient Parameters shape for the US system:", varx_results.params.shape)
print(varx_results.summary())